In [5]:
import pandas as pd
import random
from datetime import datetime, timedelta

num_pedidos = 3000
data_inicio = datetime(2025, 1, 1)
data_fim = datetime(2025, 12, 31)

produtos = {
    'Caixa Açaí Tradição 10L': 120.00,
    'Caixa Açaí 5L': 65.00,
    'Base Milkshake Baunilha 10L': 90.00,
    'Base Milkshake Morango 10L': 90.00,
    'Base Milkshake Chocolate 10L': 95.00,
    'Creme de Cupuaçu 5L': 70.00,
    'Caixa de sorvete de Creme 10L': 110.00,
    'Caixa de Sorvete de Chocolate Belga 10L': 110.00,
    'Caixa de Sorvete de Morango 10L': 110.00
}

# Clientes com ID
clientes = [{"id": 100 + i, "nome": f"Cliente_{100 + i}"} for i in range(1, 51)]
dados = []

for _ in range(num_pedidos):
    dias_totais = (data_fim - data_inicio).days
    data_pedido = data_inicio + timedelta(days=random.randint(0, dias_totais))

    if data_pedido.month in [1, 2, 3, 11, 12]:
        multiplicador_volume = random.uniform(1.5, 2.5)
    else:
        multiplicador_volume = random.uniform(0.5, 1.2)

    cliente_escolhido = random.choice(clientes)
    cliente_id = cliente_escolhido["id"]

    # Sorteio do produto GARANTIDO dentro de cada pedido
    produto = random.choice(list(produtos.keys()))
    preco_unitario = produtos[produto]

    quantidade_base = random.randint(5, 50)
    quantidade_final = int(quantidade_base * multiplicador_volume)
    quantidade_final = max(1, quantidade_final)

    valor_total = quantidade_final * preco_unitario

    dados.append([
        data_pedido.strftime("%Y-%m-%d"),
        cliente_id,
        produto,
        quantidade_final,
        round(preco_unitario, 2),
        round(valor_total, 2)
    ])

df_vendas = pd.DataFrame(dados, columns=['Data_Pedido', 'Cliente_ID', 'Produto', 'Quantidade', 'Valor_Unitario', 'Valor_Total'])
df_vendas = df_vendas.sort_values(by='Data_Pedido').reset_index(drop=True)
df_vendas.to_csv('vendas_b2b_fabrica_2025.csv', index=False)
print("Sucesso! Tabela de vendas corrigida gerada.")

Sucesso! Tabela de vendas corrigida gerada.


In [3]:
import requests
import pandas as pd

# 1. Definir os parâmetros da nossa busca na API
# Usaremos as coordenadas da região e o intervalo completo de 2025
url = "https://archive-api.open-meteo.com/v1/archive"
parametros = {
    "latitude": -23.6181,
    "longitude": -46.5567,
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "daily": "temperature_2m_mean", # Solicitando a temperatura média diária
    "timezone": "America/Sao_Paulo"
}

print("Conectando à API do Open-Meteo e extraindo dados de 2025...")

# 2. Fazendo a requisição (GET) para a API
resposta = requests.get(url, params=parametros)

# 3. Verificando se a conexão foi bem-sucedida (Status 200 significa OK)
if resposta.status_code == 200:
    dados_api = resposta.json()

    # 4. Tratando o JSON retornado para transformá-lo em uma tabela
    dados_diarios = dados_api["daily"]

    df_clima = pd.DataFrame({
        "Data_Clima": dados_diarios["time"],
        "Temperatura_Media": dados_diarios["temperature_2m_mean"]
    })

    # 5. Salvando o arquivo CSV de clima
    nome_arquivo_clima = "clima_historico_2025.csv"
    df_clima.to_csv(nome_arquivo_clima, index=False)

    print(f"Sucesso! Arquivo '{nome_arquivo_clima}' gerado com {len(df_clima)} dias de registros.")
    display(df_clima.head(10))

else:
    print(f"Erro ao conectar com a API. Código de status: {resposta.status_code}")

Conectando à API do Open-Meteo e extraindo dados de 2025...
Sucesso! Arquivo 'clima_historico_2025.csv' gerado com 365 dias de registros.


,Data_Clima,Temperatura_Media
0,2025-01-01,22.6
1,2025-01-02,24.3
2,2025-01-03,23.5
3,2025-01-04,23.1
4,2025-01-05,22.6
5,2025-01-06,22.9
6,2025-01-07,21.6
7,2025-01-08,21.4
8,2025-01-09,20.5
9,2025-01-10,20.6


In [6]:
import pandas as pd

print("Iniciando o processo de transformação e cruzamento de dados...")

# 1. Carregar os dois arquivos CSV que estão na pasta do Colab
df_vendas = pd.read_csv('vendas_b2b_fabrica_2025.csv')
df_clima = pd.read_csv('clima_historico_2025.csv')

# 2. Garantir que as colunas de data estão no mesmo formato de texto (YYYY-MM-DD)
df_vendas['Data_Pedido'] = pd.to_datetime(df_vendas['Data_Pedido']).dt.strftime('%Y-%m-%d')
df_clima['Data_Clima'] = pd.to_datetime(df_clima['Data_Clima']).dt.strftime('%Y-%m-%d')

# 3. Transformação principal: Cruzar as tabelas (Merge/Join)
# Vamos unir a tabela de vendas com a de clima onde a data do pedido for igual à data do clima
df_resultado = pd.merge(
    df_vendas,
    df_clima,
    left_on='Data_Pedido',
    right_on='Data_Clima',
    how='left'
)

# 4. Limpeza: Como já temos a data do pedido, podemos descartar a coluna duplicada 'Data_Clima'
df_resultado = df_resultado.drop(columns=['Data_Clima'])

# 5. Padronização de Negócio: Renomear as colunas para o padrão do PostgreSQL (minúsculas)
df_resultado.columns = [
    'data_pedido',
    'cliente_id',
    'produto',
    'quantidade',
    'valor_unitario',
    'valor_total',
    'temperatura_media'
]

# 6. Salvar o dataset final consolidado (Esta será a nossa Tabela Fato)
nome_arquivo_final = 'fato_vendas_clima_2025.csv'
df_resultado.to_csv(nome_arquivo_final, index=False)

print(f"Sucesso! O arquivo unificado '{nome_arquivo_final}' foi gerado com {len(df_resultado)} linhas.")
display(df_resultado.head(10))

Iniciando o processo de transformação e cruzamento de dados...
Sucesso! O arquivo unificado 'fato_vendas_clima_2025.csv' foi gerado com 3000 linhas.


,data_pedido,cliente_id,produto,quantidade,valor_unitario,valor_total,temperatura_media
0,2025-01-01,146,Caixa Açaí Tradição 10L,88,120.0,10560.0,22.6
1,2025-01-01,110,Caixa de Sorvete de Chocolate Belga 10L,28,110.0,3080.0,22.6
2,2025-01-01,128,Creme de Cupuaçu 5L,48,70.0,3360.0,22.6
3,2025-01-01,105,Base Milkshake Morango 10L,76,90.0,6840.0,22.6
4,2025-01-01,128,Caixa de sorvete de Creme 10L,80,110.0,8800.0,22.6
5,2025-01-01,118,Base Milkshake Chocolate 10L,45,95.0,4275.0,22.6
6,2025-01-01,118,Caixa de sorvete de Creme 10L,77,110.0,8470.0,22.6
7,2025-01-02,109,Base Milkshake Chocolate 10L,54,95.0,5130.0,24.3
8,2025-01-02,132,Base Milkshake Morango 10L,39,90.0,3510.0,24.3
9,2025-01-02,119,Base Milkshake Baunilha 10L,63,90.0,5670.0,24.3
